# Milestone 6 — Part 1 Extensions

This notebook implements three optional challenge extensions:
1. **Reranking** — Cross-encoder reranker with before/after comparison
2. **Hybrid Search** — Dense (ChromaDB) + Sparse (BM25) retrieval combined
3. **Query Expansion** — Automatic query reformulation before retrieval

**Prerequisites:** Run `rag_pipeline.ipynb` first to build the ChromaDB index.

In [1]:
import time
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple

import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import ollama

print('All imports successful')

All imports successful


In [2]:
# ── Reload the ChromaDB index and embedding model from Part 1 ──
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_collection('mlops_rag')

# Load all chunks from ChromaDB for BM25 index
all_data = collection.get(include=['documents', 'metadatas'])
all_chunks = all_data['documents']
all_metadata = all_data['metadatas']
all_ids = all_data['ids']

print(f'Loaded {len(all_chunks)} chunks from ChromaDB')

# Base retrieve function (from Part 1)
def retrieve_dense(query: str, k: int = 5) -> List[Dict]:
    qe = embedding_model.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=qe, n_results=k,
                               include=['documents', 'metadatas', 'distances'])
    return [{
        'rank': i+1,
        'doc_id': results['metadatas'][0][i]['doc_id'],
        'doc_title': results['metadatas'][0][i]['doc_title'],
        'content': results['documents'][0][i],
        'similarity': round(1 - results['distances'][0][i], 4),
        'chunk_id': results['ids'][0][i] if 'ids' in results else ''
    } for i in range(len(results['documents'][0]))]

# Evaluation queries with ground truth (from Part 1)
EVAL_QUERIES = [
    {'id':'Q01','query':'What is retrieval-augmented generation and how does it reduce hallucinations?','relevant_doc_ids':['doc_01']},
    {'id':'Q02','query':'What are the differences between FAISS and ChromaDB for vector search?','relevant_doc_ids':['doc_02']},
    {'id':'Q03','query':'How does data drift detection work in ML monitoring?','relevant_doc_ids':['doc_04']},
    {'id':'Q04','query':'What chunking strategies are available and what are the tradeoffs?','relevant_doc_ids':['doc_07']},
    {'id':'Q05','query':'What is vLLM and how does it improve LLM serving throughput?','relevant_doc_ids':['doc_05']},
    {'id':'Q06','query':'What metrics should be used to evaluate a RAG pipeline?','relevant_doc_ids':['doc_10','doc_01']},
    {'id':'Q07','query':'How do feature stores prevent training-serving skew?','relevant_doc_ids':['doc_06']},
    {'id':'Q08','query':'What is the best way to fine-tune a GPT model on proprietary data?','relevant_doc_ids':[]},
    {'id':'Q09','query':'What embedding model should I use for semantic search?','relevant_doc_ids':['doc_09']},
    {'id':'Q10','query':'How do agentic AI systems decide which tool to use for a given task?','relevant_doc_ids':['doc_08']}
]

def precision_at_k(retrieved, relevant, k):
    if not relevant: return 1.0 if not retrieved[:k] else 0.0
    return sum(1 for r in retrieved[:k] if r['doc_id'] in relevant) / k

def recall_at_k(retrieved, relevant, k):
    if not relevant: return 1.0
    return len(set(r['doc_id'] for r in retrieved[:k] if r['doc_id'] in relevant)) / len(relevant)

print('Setup complete')

/home/tanyasrivastava/milestone6/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 10 chunks from ChromaDB
Setup complete


---
## Extension 1: Reranking

**Approach:** Use a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to rerank the initial top-5 dense retrieval results.

**Why cross-encoders improve ranking:**
- Bi-encoders (like all-MiniLM-L6-v2) encode query and document independently — fast but less accurate
- Cross-encoders jointly encode the query+document pair, capturing fine-grained interactions — slower but more accurate
- Two-stage pipeline: retrieve top-5 cheaply with bi-encoder, then rerank top-5 accurately with cross-encoder

**Evaluation:** Compare P@1 and P@3 before vs after reranking across all 10 queries.

In [3]:
print('Loading cross-encoder reranker...')
t0 = time.time()
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f'Reranker loaded in {time.time()-t0:.2f}s')

def retrieve_with_reranking(query: str, initial_k: int = 5, final_k: int = 3) -> Tuple[List[Dict], List[Dict], float, float]:
    """
    Two-stage retrieval:
    Stage 1: Dense retrieval of top initial_k candidates (bi-encoder)
    Stage 2: Rerank candidates using cross-encoder, return top final_k
    Returns: (initial_results, reranked_results, stage1_latency_ms, stage2_latency_ms)
    """
    # Stage 1: Dense retrieval
    t0 = time.time()
    initial_results = retrieve_dense(query, k=initial_k)
    stage1_ms = (time.time() - t0) * 1000

    # Stage 2: Cross-encoder reranking
    t0 = time.time()
    pairs = [(query, r['content']) for r in initial_results]
    scores = reranker.predict(pairs)
    stage2_ms = (time.time() - t0) * 1000

    # Sort by cross-encoder score (descending)
    reranked = sorted(
        zip(initial_results, scores),
        key=lambda x: x[1],
        reverse=True
    )

    reranked_results = []
    for new_rank, (result, score) in enumerate(reranked[:final_k], 1):
        result_copy = result.copy()
        result_copy['original_rank'] = result['rank']
        result_copy['rank'] = new_rank
        result_copy['rerank_score'] = round(float(score), 4)
        reranked_results.append(result_copy)

    return initial_results[:final_k], reranked_results, stage1_ms, stage2_ms


# ── Evaluate: Before vs After Reranking ──
print('\nEvaluating reranking on 10 queries...')
print('='*70)

rerank_results = []
for eq in EVAL_QUERIES:
    initial, reranked, s1_ms, s2_ms = retrieve_with_reranking(eq['query'])

    p1_before = precision_at_k(initial, eq['relevant_doc_ids'], 1)
    p3_before = precision_at_k(initial, eq['relevant_doc_ids'], 3)
    r3_before = recall_at_k(initial, eq['relevant_doc_ids'], 3)

    p1_after = precision_at_k(reranked, eq['relevant_doc_ids'], 1)
    p3_after = precision_at_k(reranked, eq['relevant_doc_ids'], 3)
    r3_after = recall_at_k(reranked, eq['relevant_doc_ids'], 3)

    rank_change = reranked[0]['original_rank'] - 1 if reranked else 0

    rerank_results.append({
        'query_id': eq['id'],
        'query': eq['query'][:55],
        'p1_before': p1_before, 'p3_before': p3_before, 'r3_before': r3_before,
        'p1_after': p1_after,  'p3_after': p3_after,  'r3_after': r3_after,
        'stage1_ms': round(s1_ms, 1),
        'stage2_ms': round(s2_ms, 1),
        'total_ms': round(s1_ms + s2_ms, 1),
        'top1_before': initial[0]['doc_title'][:30] if initial else '',
        'top1_after': reranked[0]['doc_title'][:30] if reranked else '',
        'rank_change': rank_change
    })

df_rerank = pd.DataFrame(rerank_results)
scored = df_rerank[df_rerank['query_id'] != 'Q08']

print('\nPER-QUERY COMPARISON (Before → After Reranking):')
print(f'{"Query":6} | {"P@1 Before":10} | {"P@1 After":10} | {"P@3 Before":10} | {"P@3 After":10} | {"Rank Change":12} | {"Latency ms"}')
print('-'*80)
for r in rerank_results:
    change_str = f"rank {r['rank_change']+1}→1" if r['rank_change'] > 0 else 'no change'
    print(f"{r['query_id']:6} | {r['p1_before']:10.2f} | {r['p1_after']:10.2f} | {r['p3_before']:10.2f} | {r['p3_after']:10.2f} | {change_str:12} | +{r['stage2_ms']:.0f}ms")

print(f'\nAGGREGATE (excluding out-of-scope Q08):')
print(f"Mean P@1 Before: {scored['p1_before'].mean():.3f}  →  After: {scored['p1_after'].mean():.3f}")
print(f"Mean P@3 Before: {scored['p3_before'].mean():.3f}  →  After: {scored['p3_after'].mean():.3f}")
print(f"Mean R@3 Before: {scored['r3_before'].mean():.3f}  →  After: {scored['r3_after'].mean():.3f}")
print(f"Mean reranker latency: {df_rerank['stage2_ms'].mean():.1f}ms")
print(f"Mean total latency:    {df_rerank['total_ms'].mean():.1f}ms")

Loading cross-encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded in 1.96s

Evaluating reranking on 10 queries...

PER-QUERY COMPARISON (Before → After Reranking):
Query  | P@1 Before | P@1 After  | P@3 Before | P@3 After  | Rank Change  | Latency ms
--------------------------------------------------------------------------------
Q01    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +793ms
Q02    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +477ms
Q03    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +434ms
Q04    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +461ms
Q05    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +458ms
Q06    |       1.00 |       1.00 |       0.67 |       0.67 | no change    | +434ms
Q07    |       1.00 |       1.00 |       0.33 |       0.33 | no change    | +450ms
Q08    |       0.00 |       0.00 |       0.00 |       0.00 | no change    | +434ms
Q09    |       1.00 |       1.00 |       0.33 |       

---
## Extension 2: Hybrid Search

**Approach:** Combine dense retrieval (ChromaDB cosine similarity) with sparse retrieval (BM25 keyword matching) using Reciprocal Rank Fusion (RRF).

**Why hybrid search helps:**
- Dense retrieval excels at semantic/conceptual queries (e.g., 'what reduces hallucinations')
- Sparse BM25 excels at exact keyword matches (e.g., 'PagedAttention', 'Kolmogorov-Smirnov')
- RRF fusion combines both rankings without requiring score normalization
- RRF formula: score(d) = Σ 1/(k + rank(d)) where k=60 is a smoothing constant

In [4]:
# Build BM25 sparse index over all chunks
print('Building BM25 sparse index...')
t0 = time.time()
tokenized_corpus = [chunk.lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)
print(f'BM25 index built in {time.time()-t0:.3f}s over {len(all_chunks)} chunks')


def retrieve_sparse(query: str, k: int = 5) -> List[Dict]:
    """BM25 sparse retrieval."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:k]
    return [{
        'rank': i+1,
        'doc_id': all_metadata[idx]['doc_id'],
        'doc_title': all_metadata[idx]['doc_title'],
        'content': all_chunks[idx],
        'bm25_score': round(float(scores[idx]), 4),
        'chunk_index': idx
    } for i, idx in enumerate(top_indices)]


def reciprocal_rank_fusion(dense_results: List[Dict], sparse_results: List[Dict],
                           k: int = 60, top_n: int = 3) -> List[Dict]:
    """
    Reciprocal Rank Fusion (RRF) combines dense and sparse rankings.
    score(d) = 1/(k + rank_dense(d)) + 1/(k + rank_sparse(d))
    k=60 is the standard smoothing constant from the original RRF paper.
    """
    rrf_scores = {}
    doc_info = {}

    for r in dense_results:
        key = r['chunk_id'] if 'chunk_id' in r else r['doc_id'] + str(r.get('rank', 0))
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + r['rank'])
        doc_info[key] = r

    for r in sparse_results:
        key = all_ids[r['chunk_index']] if 'chunk_index' in r else r['doc_id'] + str(r.get('rank', 0))
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + r['rank'])
        if key not in doc_info:
            doc_info[key] = r

    sorted_keys = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_n]

    fused = []
    for new_rank, key in enumerate(sorted_keys, 1):
        item = doc_info[key].copy()
        item['rank'] = new_rank
        item['rrf_score'] = round(rrf_scores[key], 6)
        fused.append(item)
    return fused


def retrieve_hybrid(query: str, k: int = 3) -> Tuple[List[Dict], List[Dict], List[Dict], float]:
    """Full hybrid retrieval: dense + sparse + RRF fusion."""
    t0 = time.time()
    dense = retrieve_dense(query, k=5)
    sparse = retrieve_sparse(query, k=5)
    fused = reciprocal_rank_fusion(dense, sparse, top_n=k)
    latency_ms = (time.time() - t0) * 1000
    return dense[:k], sparse[:k], fused, latency_ms


# ── Evaluate: Dense vs Sparse vs Hybrid ──
print('\nEvaluating hybrid search on 10 queries...')
print('='*70)

hybrid_results = []
for eq in EVAL_QUERIES:
    dense, sparse, fused, lat = retrieve_hybrid(eq['query'])

    hybrid_results.append({
        'query_id': eq['id'],
        'p1_dense':  precision_at_k(dense,  eq['relevant_doc_ids'], 1),
        'p1_sparse': precision_at_k(sparse, eq['relevant_doc_ids'], 1),
        'p1_hybrid': precision_at_k(fused,  eq['relevant_doc_ids'], 1),
        'p3_dense':  precision_at_k(dense,  eq['relevant_doc_ids'], 3),
        'p3_sparse': precision_at_k(sparse, eq['relevant_doc_ids'], 3),
        'p3_hybrid': precision_at_k(fused,  eq['relevant_doc_ids'], 3),
        'r3_dense':  recall_at_k(dense,  eq['relevant_doc_ids'], 3),
        'r3_sparse': recall_at_k(sparse, eq['relevant_doc_ids'], 3),
        'r3_hybrid': recall_at_k(fused,  eq['relevant_doc_ids'], 3),
        'latency_ms': round(lat, 1)
    })

df_hybrid = pd.DataFrame(hybrid_results)
scored_h = df_hybrid[df_hybrid['query_id'] != 'Q08']

print(f'\n{"Query":6} | {"P@1 Dense":10} | {"P@1 Sparse":10} | {"P@1 Hybrid":10}')
print('-'*50)
for r in hybrid_results:
    print(f"{r['query_id']:6} | {r['p1_dense']:10.2f} | {r['p1_sparse']:10.2f} | {r['p1_hybrid']:10.2f}")

print(f'\nAGGREGATE COMPARISON (excluding Q08):')
print(f"         {'Dense':>8} {'Sparse':>8} {'Hybrid':>8}")
print(f"Mean P@1: {scored_h['p1_dense'].mean():8.3f} {scored_h['p1_sparse'].mean():8.3f} {scored_h['p1_hybrid'].mean():8.3f}")
print(f"Mean P@3: {scored_h['p3_dense'].mean():8.3f} {scored_h['p3_sparse'].mean():8.3f} {scored_h['p3_hybrid'].mean():8.3f}")
print(f"Mean R@3: {scored_h['r3_dense'].mean():8.3f} {scored_h['r3_sparse'].mean():8.3f} {scored_h['r3_hybrid'].mean():8.3f}")
print(f"Mean latency: {df_hybrid['latency_ms'].mean():.1f}ms")

Building BM25 sparse index...
BM25 index built in 0.003s over 10 chunks

Evaluating hybrid search on 10 queries...

Query  | P@1 Dense  | P@1 Sparse | P@1 Hybrid
--------------------------------------------------
Q01    |       1.00 |       1.00 |       1.00
Q02    |       1.00 |       1.00 |       1.00
Q03    |       1.00 |       1.00 |       1.00
Q04    |       1.00 |       1.00 |       1.00
Q05    |       1.00 |       1.00 |       1.00
Q06    |       1.00 |       1.00 |       1.00
Q07    |       1.00 |       1.00 |       1.00
Q08    |       0.00 |       0.00 |       0.00
Q09    |       1.00 |       0.00 |       1.00
Q10    |       1.00 |       1.00 |       1.00

AGGREGATE COMPARISON (excluding Q08):
            Dense   Sparse   Hybrid
Mean P@1:    1.000    0.889    1.000
Mean P@3:    0.370    0.370    0.370
Mean R@3:    1.000    1.000    1.000
Mean latency: 19.9ms


---
## Extension 3: Query Expansion

**Approach:** Use Mistral 7B to automatically rewrite/expand the original query into a more specific and information-rich query before retrieval.

**Why query expansion helps:**
- Short or ambiguous queries (e.g., 'What embedding model should I use?') may not match document vocabulary well
- The LLM adds domain-specific terminology, synonyms, and related concepts
- Expanded queries produce better embedding representations that align more closely with document content

**Evaluation:** Compare P@1 and P@3 before vs after expansion across all 10 queries.

In [5]:
def expand_query(query: str) -> Tuple[str, float]:
    """
    Rewrites the query using Mistral 7B to be more specific and information-rich.
    Adds relevant technical terminology and related concepts.
    Returns: (expanded_query, latency_ms)
    """
    prompt = f"""You are a search query optimizer for a technical MLOps knowledge base.
Rewrite the following query to be more specific and detailed.
Add relevant technical terms, synonyms, and related concepts that would help find relevant documents.
Return ONLY the rewritten query as a single sentence. No explanations.

Original query: {query}

Rewritten query:"""

    t0 = time.time()
    response = ollama.chat(
        model='mistral:7b-instruct',
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.2, 'num_predict': 80}
    )
    latency_ms = (time.time() - t0) * 1000
    expanded = response['message']['content'].strip()
    # Clean up any quotes or prefixes the model might add
    expanded = expanded.strip('"').strip("'").strip()
    return expanded, latency_ms


def retrieve_with_expansion(query: str, k: int = 3) -> Tuple[List[Dict], List[Dict], str, float, float]:
    """
    Full query expansion pipeline:
    1. Expand query using LLM
    2. Retrieve using expanded query
    Returns: (original_results, expanded_results, expanded_query, expansion_ms, retrieval_ms)
    """
    # Original retrieval
    original_results = retrieve_dense(query, k=k)

    # Expand the query
    expanded_query, expansion_ms = expand_query(query)

    # Retrieve with expanded query
    t0 = time.time()
    expanded_results = retrieve_dense(expanded_query, k=k)
    retrieval_ms = (time.time() - t0) * 1000

    return original_results, expanded_results, expanded_query, expansion_ms, retrieval_ms


# ── Evaluate: Original vs Expanded Query Retrieval ──
print('Evaluating query expansion on 10 queries... (takes ~2 minutes)')
print('='*70)

expansion_results = []
for eq in EVAL_QUERIES:
    print(f"  {eq['id']}: {eq['query'][:50]}...")
    orig, expanded_res, expanded_q, exp_ms, ret_ms = retrieve_with_expansion(eq['query'])

    expansion_results.append({
        'query_id': eq['id'],
        'original_query': eq['query'],
        'expanded_query': expanded_q,
        'p1_original': precision_at_k(orig, eq['relevant_doc_ids'], 1),
        'p3_original': precision_at_k(orig, eq['relevant_doc_ids'], 3),
        'r3_original': recall_at_k(orig,  eq['relevant_doc_ids'], 3),
        'p1_expanded': precision_at_k(expanded_res, eq['relevant_doc_ids'], 1),
        'p3_expanded': precision_at_k(expanded_res, eq['relevant_doc_ids'], 3),
        'r3_expanded': recall_at_k(expanded_res,    eq['relevant_doc_ids'], 3),
        'expansion_ms': round(exp_ms, 1),
        'retrieval_ms': round(ret_ms, 1)
    })
    print(f"    Expanded: {expanded_q[:80]}")
    print(f"    P@1: {expansion_results[-1]['p1_original']:.2f} → {expansion_results[-1]['p1_expanded']:.2f}")

df_exp = pd.DataFrame(expansion_results)
scored_e = df_exp[df_exp['query_id'] != 'Q08']

print(f'\nAGGREGATE COMPARISON (excluding Q08):')
print(f"           {'Original':>10} {'Expanded':>10}")
print(f"Mean P@1:  {scored_e['p1_original'].mean():10.3f} {scored_e['p1_expanded'].mean():10.3f}")
print(f"Mean P@3:  {scored_e['p3_original'].mean():10.3f} {scored_e['p3_expanded'].mean():10.3f}")
print(f"Mean R@3:  {scored_e['r3_original'].mean():10.3f} {scored_e['r3_expanded'].mean():10.3f}")
print(f"Mean expansion latency: {df_exp['expansion_ms'].mean():.0f}ms")

print('\nQUERY EXPANSION EXAMPLES:')
for r in expansion_results:
    print(f"\n{r['query_id']}:")
    print(f"  Original: {r['original_query']}")
    print(f"  Expanded: {r['expanded_query']}")
    improvement = r['p1_expanded'] - r['p1_original']
    print(f"  P@1 change: {improvement:+.2f}")

Evaluating query expansion on 10 queries... (takes ~2 minutes)
  Q01: What is retrieval-augmented generation and how doe...
    Expanded: Explain Retrieval-Augmented Generation in MLOps, focusing on its role in mitigat
    P@1: 1.00 → 1.00
  Q02: What are the differences between FAISS and ChromaD...
    Expanded: Differentiate the key technical aspects of Faiss and ChromaDB in terms of their 
    P@1: 1.00 → 1.00
  Q03: How does data drift detection work in ML monitorin...
    Expanded: Explain the mechanism of anomaly detection for data drift in machine learning pi
    P@1: 1.00 → 1.00
  Q04: What chunking strategies are available and what ar...
    Expanded: Explore various data chunking techniques for machine learning pipeline optimizat
    P@1: 1.00 → 0.00
  Q05: What is vLLM and how does it improve LLM serving t...
    Expanded: What is Vertex AI Large Language Model (vLLM) and how does it enhance the servin
    P@1: 1.00 → 1.00
  Q06: What metrics should be used to evaluate a RAG

In [6]:
import pandas as pd
scored_r = df_rerank[df_rerank["query_id"] != "Q08"]
scored_h = df_hybrid[df_hybrid["query_id"] != "Q08"]
scored_e = df_exp[df_exp["query_id"] != "Q08"]

rerank_lat = df_rerank["stage2_ms"].mean()
exp_lat = df_exp["expansion_ms"].mean()

print("="*70)
print("EXTENSIONS SUMMARY: All Three Approaches")
print("="*70)
print(f"\n{'Method':<20} {'P@1':>8} {'P@3':>8} {'R@3':>8} {'Extra Latency':>15}")
print("-"*60)
print(f"{'Baseline (dense)':<20} {scored_r['p1_before'].mean():8.3f} {scored_r['p3_before'].mean():8.3f} {scored_r['r3_before'].mean():8.3f} {'~31ms':>15}")
print(f"{'+ Reranking':<20} {scored_r['p1_after'].mean():8.3f} {scored_r['p3_after'].mean():8.3f} {scored_r['r3_after'].mean():8.3f} {('+'+ str(round(rerank_lat))+'ms'):>15}")
print(f"{'+ Hybrid (RRF)':<20} {scored_h['p1_hybrid'].mean():8.3f} {scored_h['p3_hybrid'].mean():8.3f} {scored_h['r3_hybrid'].mean():8.3f} {'<5ms':>15}")
print(f"{'+ Query Expansion':<20} {scored_e['p1_expanded'].mean():8.3f} {scored_e['p3_expanded'].mean():8.3f} {scored_e['r3_expanded'].mean():8.3f} {('+'+ str(round(exp_lat))+'ms'):>15}")

ext_summary = {
    "reranking": df_rerank.to_dict("records"),
    "hybrid_search": df_hybrid.to_dict("records"),
    "query_expansion": df_exp.to_dict("records")
}
import json
with open("extension_results.json", "w") as f:
    json.dump(ext_summary, f, indent=2)
print("\nResults saved to extension_results.json")


SyntaxError: invalid decimal literal (2590157471.py, line 13)